# Spatial Studio · Kaggle GPU feasibility test
**Goal:** verify GPU, COLMAP, and availability of a reconstruction pipeline without paid hosting. This notebook does **not** connect Kaggle to the website and does not promise a final Gaussian Splat model.

1. In Kaggle, choose **File → Import Notebook** and import this `.ipynb` from GitHub/download.
2. Enable **Settings → Accelerator → GPU** and **Internet on** if available.
3. Optionally attach a private Kaggle Dataset containing a short room video; do not make private room footage public.
4. Run all cells. The report is saved under `/kaggle/working/spatial_gpu_report.json`.


In [ ]:
import os, sys, subprocess, json, shutil, pathlib, platform, time
from pathlib import Path
out=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
report={'platform':platform.platform(),'python':sys.version.split()[0]}
def command(args, timeout=120):
    try:
        p=subprocess.run(args,capture_output=True,text=True,timeout=timeout)
        return {'returncode':p.returncode,'stdout':p.stdout[-3000:],'stderr':p.stderr[-1500:]}
    except Exception as e:return {'error':str(e)}
report['gpu']=command(['nvidia-smi']) if shutil.which('nvidia-smi') else {'error':'nvidia-smi missing'}
print(json.dumps(report['gpu'],ensure_ascii=False,indent=2))


In [ ]:
# Check installed utilities. No system modification.
for tool in ['ffmpeg','ffprobe','colmap','git','nvcc']:
    report[tool]={'path':shutil.which(tool)}
    if shutil.which(tool):report[tool]['version']=command([tool,'-version'] if tool!='git' else [tool,'--version'],timeout=20)
print(json.dumps({k:report[k]['path'] for k in ['ffmpeg','ffprobe','colmap','git','nvcc']},indent=2))


In [ ]:
# Discover attached video, if any. Kaggle datasets are mounted under /kaggle/input.
video_extensions={'.mp4','.mov','.m4v','.webm'}
videos=[p for p in Path('/kaggle/input').rglob('*') if p.is_file() and p.suffix.lower() in video_extensions] if Path('/kaggle/input').exists() else []
report['videos']=[{'path':str(p),'size_mb':round(p.stat().st_size/1048576,2)} for p in videos[:10]]
print('Videos found:',len(videos))
print(json.dumps(report['videos'],indent=2))


In [ ]:
# Extract 1 frame per second from first video to validate ingestion. This is NOT 3D reconstruction.
frames=out/'spatial_frames';frames.mkdir(exist_ok=True)
if videos and shutil.which('ffmpeg'):
    report['extract']=command(['ffmpeg','-hide_banner','-y','-i',str(videos[0]),'-vf','fps=1,scale=960:-2','-frames:v','30',str(frames/'frame_%04d.jpg')],timeout=180)
    report['frame_count']=len(list(frames.glob('*.jpg')))
    print('Extracted frames:',report['frame_count'])
else:
    report['extract']={'skipped':'Attach a private video dataset and ensure ffmpeg is installed'}
    print(report['extract'])


In [ ]:
# Save an explicit machine-readable report; no fake reconstruction output.
report['colmap_ready']=bool(shutil.which('colmap'))
report['brush_ready']=bool(shutil.which('brush_app'))
report['reconstruction_completed']=False
report['next_step']='Install and validate compatible COLMAP + Brush build before full GPU reconstruction; Kaggle sessions are ephemeral.'
path=out/'spatial_gpu_report.json';path.write_text(json.dumps(report,ensure_ascii=False,indent=2))
print('Report:',path)
print('COLMAP installed:',report['colmap_ready'],'Brush installed:',report['brush_ready'])
